# Sentiment analysis 

The objective of the second problem is to perform Sentiment analysis from the tweets collected from the users targeted at various mobile devices.
Based on the tweet posted by a user (text), we will classify if the sentiment of the user targeted at a particular mobile device is positive or not.

In [453]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Question 1

### Read the data
- read tweets.csv
- use latin encoding if it gives encoding error while loading

In [454]:
import numpy as np
import pandas as pd

tweet_df = pd.read_csv("/content/drive/My Drive/GreatLakes/NLP/tweets.csv", encoding='latin1')
tweet_df.shape

(9093, 3)

In [455]:
tweet_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9093 entries, 0 to 9092
Data columns (total 3 columns):
tweet_text                                            9092 non-null object
emotion_in_tweet_is_directed_at                       3291 non-null object
is_there_an_emotion_directed_at_a_brand_or_product    9093 non-null object
dtypes: object(3)
memory usage: 213.2+ KB


### Drop null values
- drop all the rows with null values

In [0]:
tweet_df.dropna(inplace=True)

In [457]:
tweet_df.shape

(3291, 3)

### Print the dataframe
- print initial 5 rows of the data
- use df.head()

In [458]:
tweet_df.head()

,tweet_text,emotion_in_tweet_is_directed_at,is_there_an_emotion_directed_at_a_brand_or_product
0,.@wesley83 I have a 3G iPhone. After 3 hrs twe...,iPhone,Negative emotion
1,@jessedee Know about @fludapp ? Awesome iPad/i...,iPad or iPhone App,Positive emotion
2,@swonderlin Can not wait for #iPad 2 also. The...,iPad,Positive emotion
3,@sxsw I hope this year's festival isn't as cra...,iPad or iPhone App,Negative emotion
4,@sxtxstate great stuff on Fri #SXSW: Marissa M...,Google,Positive emotion


## Question 2

### Preprocess data
- convert all text to lowercase - use .lower()
- select only numbers, alphabets, and #+_ from text - use re.sub()
- strip all the text - use .strip()
    - this is for removing extra spaces

In [0]:
#Applying lower() to all the columns so that handling is easier
tweet_df = tweet_df.apply(lambda x: x.str.lower())

In [0]:
import re
tweet_df['tweet_text'] = tweet_df['tweet_text'].apply(lambda x: re.sub("[^\w\+# ]", "", x))

In [0]:
#Strip the spaces from all the column texts
tweet_df = tweet_df.apply(lambda x: x.str.strip())

print dataframe

In [462]:
print(tweet_df)

                                             tweet_text  ... is_there_an_emotion_directed_at_a_brand_or_product
0     wesley83 i have a 3g iphone after 3 hrs tweeti...  ...                                   negative emotion
1     jessedee know about fludapp  awesome ipadiphon...  ...                                   positive emotion
2     swonderlin can not wait for #ipad 2 also they ...  ...                                   positive emotion
3     sxsw i hope this years festival isnt as crashy...  ...                                   negative emotion
4     sxtxstate great stuff on fri #sxsw marissa may...  ...                                   positive emotion
...                                                 ...  ...                                                ...
9077  mention your pr guy just convinced me to switc...  ...                                   positive emotion
9079  quotpapyrussort of like the ipadquot  nice lol...  ...                                   positive 

## Question 3

### Preprocess data
- in column "is_there_an_emotion_directed_at_a_brand_or_product"
    - select only those rows where value equal to "positive emotion" or "negative emotion"
- find the value counts of "positive emotion" and "negative emotion"

In [463]:
tweet_df["is_there_an_emotion_directed_at_a_brand_or_product"].value_counts()

positive emotion                      2672
negative emotion                       519
no emotion toward brand or product      91
i can't tell                             9
Name: is_there_an_emotion_directed_at_a_brand_or_product, dtype: int64

In [0]:
tweet_df = tweet_df[tweet_df["is_there_an_emotion_directed_at_a_brand_or_product"].isin(["positive emotion","negative emotion"])] 

In [465]:
tweet_df["is_there_an_emotion_directed_at_a_brand_or_product"].value_counts()

positive emotion    2672
negative emotion     519
Name: is_there_an_emotion_directed_at_a_brand_or_product, dtype: int64

## Question 4

### Encode labels
- in column "is_there_an_emotion_directed_at_a_brand_or_product"
    - change "positive emotion" to 1
    - change "negative emotion" to 0
- use map function to replace values

In [466]:
tweet_df['is_there_an_emotion_directed_at_a_brand_or_product'] = tweet_df['is_there_an_emotion_directed_at_a_brand_or_product'].map({'positive emotion':1, 'negative emotion':0})
tweet_df["is_there_an_emotion_directed_at_a_brand_or_product"].value_counts()

1    2672
0     519
Name: is_there_an_emotion_directed_at_a_brand_or_product, dtype: int64

In [0]:
#Observation : Positive emotion row count is almost 5 times the negative emotion row count. Positive emotion comprises 83.7% of complete dataset

## Question 5

### Get feature and label
- get column "tweet_text" as feature
- get column "is_there_an_emotion_directed_at_a_brand_or_product" as label

In [0]:
feature = tweet_df["tweet_text"]
label = tweet_df["is_there_an_emotion_directed_at_a_brand_or_product"]

### Create train and test data
- use train_test_split to get train and test set
- set a random_state
- test_size: 0.25

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(feature, label, test_size=0.25, random_state=7)

In [470]:
print(X_train.shape, y_train.shape)

(2393,) (2393,)


In [471]:
print(X_test.shape, y_test.shape)

(798,) (798,)


## Question 6

### Vectorize data
- create document-term matrix
- use CountVectorizer()
    - ngram_range: (1, 2)
    - stop_words: 'english'
    - min_df: 2   
- do fit_transform on X_train
- do transform on X_test

In [0]:
#We will use CountVectorizer to "convert text into a matrix of token counts":
from sklearn.feature_extraction.text import CountVectorizer
vect_data = CountVectorizer(ngram_range = (1,2), stop_words='english', min_df=2)

In [0]:
X_train_dtm = vect_data.fit_transform(X_train)

In [0]:
X_test_dtm = vect_data.transform(X_test)

## Question 7

### Select classifier logistic regression
- use logistic regression for predicting sentiment of the given tweet
- initialize classifier

In [0]:
from sklearn.linear_model import LogisticRegression
logistic_classifier = LogisticRegression()

### Fit the classifer
- fit logistic regression classifier

In [476]:
logistic_classifier.fit(X_train_dtm, y_train)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=100,
                   multi_class='auto', n_jobs=None, penalty='l2',
                   random_state=None, solver='lbfgs', tol=0.0001, verbose=0,
                   warm_start=False)

## Question 8

### Select classifier naive bayes
- use naive bayes for predicting sentiment of the given tweet
- initialize classifier
- use MultinomialNB

In [0]:
from sklearn.naive_bayes import MultinomialNB

nb_classifier = MultinomialNB()

### Fit the classifer
- fit naive bayes classifier

In [478]:
nb_classifier.fit(X_train_dtm, y_train)

MultinomialNB(alpha=1.0, class_prior=None, fit_prior=True)

## Question 9

### Make predictions on logistic regression
- use your trained logistic regression model to make predictions on X_test

In [0]:
y_pred_lclabel = logistic_classifier.predict(X_test_dtm)

### Make predictions on naive bayes
- use your trained naive bayes model to make predictions on X_test
- use a different variable name to store predictions so that they are kept separately

In [0]:
y_pred_nblabel = nb_classifier.predict(X_test_dtm)

## Question 10

### Calculate accuracy of logistic regression
- check accuracy of logistic regression classifer
- use sklearn.metrics.accuracy_score

In [481]:
from sklearn import metrics

metrics.accuracy_score(y_test, y_pred_lclabel)

0.87468671679198

### Calculate accuracy of naive bayes
- check accuracy of naive bayes classifer
- use sklearn.metrics.accuracy_score

In [482]:
metrics.accuracy_score(y_test, y_pred_nblabel)

0.8671679197994987

**Comparison:
Accuracy score with both naive bayes and logistic regression is almost same (Logistic Regression being slightly better than Naive Bayes). 

Since the data is highly imbalanced so accuracy is biased towards high count
positive outcome class i.e. positive emotion, A dumb model will have accuracy of around 84%. So classification accuracy given by models above is slightly better than that of dumb (0 model) accuracy. 

Let us calculate AUC to see which model performs better**


**ADDITIONAL(FOR REFERENCE): COMPARING USING CONFUSION MATRIX AUC SCORE**

**Confusion Matrix 
Since more data sample is of positive emotion(1). If we treat negative emotion(0) as class of interest to see how we can improve our product. Let us see which model performs better for 0 class**

In [483]:
#Linear classifier confusion matrix
print(metrics.confusion_matrix(y_test, y_pred_lclabel))
#TN = 41, FN= 14

[[ 41  86]
 [ 14 657]]


In [484]:
#Naive bayes classifier confusion matrix
print(metrics.confusion_matrix(y_test, y_pred_nblabel))
#TN= 39, FN = 18

[[ 39  88]
 [ 18 653]]


In [0]:
#Observation: Linear classifier is performing better for True Negative cases (i.e. identifying more True Negatives than Naive Bayes model)

*AUC is useful as a single number summary of classifier performance
(Higher AUC value = better classifier)
If you randomly chose one positive and one negative observation, AUC represents the likelihood that your classifier will assign a higher predicted probability to the positive observation
AUC is useful even when there is high class imbalance (unlike classification accuracy)*


In [0]:
# calculate predicted probabilities for X_test_dtm
y_pred_prob_lc = logistic_classifier.predict_proba(X_test_dtm)[:, 1]
y_pred_prob_nb = nb_classifier.predict_proba(X_test_dtm)[:, 1]

In [487]:
# calculate AUC
print("AUC for logistic regression",metrics.roc_auc_score(y_test, y_pred_prob_lc))

AUC for logistic regression 0.8864311111632656


In [488]:
print("AUC for Naive bayes",metrics.roc_auc_score(y_test, y_pred_prob_nb))

AUC for Naive bayes 0.8386706877735663


In [0]:
#Conclusion:
#Logistic Regression performance is better in terms of accuracy score, predicatbility of Negative sentiments and AUC despite the data being imbalanced